# Week 1 — Task 5: A Real Time Series

**Name:** Tanzeela Awan  
**Roll No:** SU92-BSDSM-S23-002  

**Dataset:** `flights.csv` — 144 rows, monthly airline passenger totals 1949–1960.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('data/flights.csv')
print(f'Shape: {df.shape}')
print(f'Years: {df["year"].min()} to {df["year"].max()}')
print(f'Months: {df["month"].unique()}')
print(f'Missing values: {df.isnull().sum().sum()}')
df.head()

## Step 1 — Full Time Series

Combine `year` and `month` into a single ordered date axis.

In [ ]:
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
df['month_num'] = df['month'].apply(lambda m: month_order.index(m) + 1)
df['date'] = pd.to_datetime(
    df['year'].astype(str) + '-' + df['month_num'].astype(str).str.zfill(2) + '-01'
)
df = df.sort_values('date').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(df['date'], df['passengers'], lw=1.8, color='#2166ac')
ax.set_title(
    'Airline passengers grew from 112K (Jan 1949) to 432K (Dec 1960)\n'
    'with a clear repeating seasonal cycle overlaid on the growth trend',
    loc='left'
)
ax.set_xlabel('Date')
ax.set_ylabel('Passengers (thousands)')
ax.set_ylim(0, 700)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('task5_full_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 2 — Isolating the Seasonal Cycle (One Line Per Year)

In [ ]:
years = sorted(df['year'].unique())
year_palette = plt.cm.Blues(np.linspace(0.3, 0.95, len(years)))

fig, ax = plt.subplots(figsize=(11, 6))
for i, yr in enumerate(years):
    sub = df[df['year'] == yr].sort_values('month_num')
    ax.plot(sub['month_num'], sub['passengers'], lw=1.8,
            color=year_palette[i], label=str(yr))

ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_order)
ax.set_title(
    'Seasonal shape is consistent every year — peak in July/August, trough in February\n'
    '(darker = later year; all 12 years shown)',
    loc='left'
)
ax.set_xlabel('Month')
ax.set_ylabel('Passengers (thousands)')
ax.set_ylim(0, 700)
ax.legend(title='Year', frameon=False, ncol=2, fontsize=8)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('task5_seasonal_lines.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3 — Which Month is Consistently Busiest?

In [ ]:
monthly_avg = (
    df.groupby('month_num')['passengers']
    .mean()
    .reset_index()
)
monthly_avg['month_label'] = monthly_avg['month_num'].apply(lambda n: month_order[n-1])

peak_months = {7, 8}  # July and August
colors_bar = ['#d73027' if m in peak_months else '#4575b4'
              for m in monthly_avg['month_num']]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(monthly_avg['month_label'], monthly_avg['passengers'],
       color=colors_bar, alpha=0.85, edgecolor='white')
ax.set_title(
    'July and August are consistently the busiest months across all 12 years (highlighted in red)',
    loc='left'
)
ax.set_ylabel('Average passengers (thousands)')
ax.set_ylim(0, 450)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('task5_busiest_month.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4 — Which Chart Goes in a Report?

| Chart | Question it answers | Include in report? |
|---|---|---|
| Chart 1 (full time series) | How large is the airline and how fast did it grow? | **Yes** — if the report is about long-term growth |
| Chart 2 (seasonal lines) | What is the seasonal shape, and is it stable? | **Yes** — if the report is about seasonality |
| Chart 3 (monthly bar) | Which month is consistently busiest? | **Yes, in place of Chart 2** — if the only question is which month |

Charts 2 and 3 answer similar questions at different levels of detail. In a single report, **Chart 2 is richer** (it also shows year-over-year comparisons) and Chart 3 is a subset of it. I would put Chart 1 and Chart 2 in the same report, and omit Chart 3 — it answers the same question as Chart 2 with less information.

## Answer: Is the Seasonal Swing Growing or Just the Airline?

From Chart 2, the absolute swing (July peak minus February trough) grows visibly larger each year — from roughly 40K passengers in 1949 to roughly 140K in 1960. This *could* be because:

- **(a)** Seasonality is genuinely intensifying (a growing fraction of travel happens in summer), or  
- **(b)** The airline is simply growing, and the same proportional seasonal pattern produces a bigger absolute swing.

**Chart to settle it:** Plot the **seasonal index** — each month's passengers as a percentage of that year's annual mean (or total). If the July index stays roughly constant at, say, 130% every year, the pattern is proportional (option b). If the July percentage rises year over year, the seasonality itself is intensifying (option a).

A cursory inspection of Chart 2 suggests option (b) is more likely — the *shape* of the seasonal curve looks similar each year, just scaled up. But the chart that would confirm this is the seasonal-index version.

In [ ]:
# Bonus: seasonal index — does the shape stay constant or intensify?
annual_mean = df.groupby('year')['passengers'].transform('mean')
df['seasonal_index'] = (df['passengers'] / annual_mean) * 100

fig, ax = plt.subplots(figsize=(11, 6))
for i, yr in enumerate(years):
    sub = df[df['year'] == yr].sort_values('month_num')
    ax.plot(sub['month_num'], sub['seasonal_index'], lw=1.5,
            color=year_palette[i], alpha=0.85, label=str(yr))

ax.axhline(100, color='#888888', lw=1, linestyle='--')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_order)
ax.set_title(
    'Seasonal index (passengers as % of annual mean) is near-stable — growth explains the widening swing, not stronger seasonality',
    loc='left'
)
ax.set_xlabel('Month')
ax.set_ylabel('Seasonal index (annual mean = 100)')
ax.legend(title='Year', frameon=False, ncol=2, fontsize=8)
for sp in ('top', 'right'):
    ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('task5_seasonal_index.png', dpi=150, bbox_inches='tight')
plt.show()
print('The seasonal index lines cluster tightly — the proportional shape is stable across years.')
print('Conclusion: the wider absolute swing is a consequence of airline growth, not intensifying seasonality.')